# 02 负荷规律分析与分区用电特征对比

**题目四 城市电力负荷分析与预测** ｜ 负责：江玉婷（GitHub: Lizhuiii；规律分析 + 报告主笔）

| 项目 | 内容 |
|---|---|
| 上游数据 | `data/processed/power_load_cleaned.csv`（01 清洗交付，本 notebook 不再改清洗规则） |
| 本 notebook 产出 | 日 / 周 / 季节负荷规律、三分区用电特征、统计表与报告图 1—6（04 另产出报告图 7—9） |
| 对应题目要求 | 分析负荷在日、周、季节尺度上的变化规律，并比较三个分区的用电特征 |
| 下游消费者 | 谭煜霖（成品图重绘）、江玉婷（报告第 4-5 节） |

## 数据口径

- 原始数据为 10 分钟采样，清洗后共 52,416 条。
- 可观测区间为 2017-01-01 00:00 至 2017-12-30 23:50，共 **364 天**；文件中没有 2017-12-31。
- 因此报告统一写“2017 年可观测的 364 天”，不写“全年 365 天”。
- 所有小时、星期、月份、季节均值都按现有样本等权平均；不把缺失的 12 月 31 日补成虚构观测。

## 步骤 0　环境与路径


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)


def find_root(marker="data"):
    "从当前工作目录向上查找仓库根目录（与 01 保持一致）。"
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / marker / "raw").is_dir():
            return d
    raise FileNotFoundError(f"未在 {here} 及其父目录中找到 {marker}/raw")


ROOT = find_root()
PROCESSED = ROOT / "data" / "processed" / "power_load_cleaned.csv"
FIG_DIR = ROOT / "figures" / "patterns"
TABLE_DIR = ROOT / "tables"
for p in (FIG_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

ZONE_COLS = ["zone1_power_consumption", "zone2_power_consumption", "zone3_power_consumption"]
TOTAL = "zone_total_power_consumption"
COL_LABEL = {TOTAL: "总负荷", ZONE_COLS[0]: "Zone1", ZONE_COLS[1]: "Zone2", ZONE_COLS[2]: "Zone3"}
COL_COLOR = {TOTAL: "#1f4e9c", ZONE_COLS[0]: "#e07b39", ZONE_COLS[1]: "#2ca25f", ZONE_COLS[2]: "#c0392b"}
SEASON_ORDER = ["春季", "夏季", "秋季", "冬季"]
SEASON_MAP = {
    12: "冬季", 1: "冬季", 2: "冬季",
    3: "春季", 4: "春季", 5: "春季",
    6: "夏季", 7: "夏季", 8: "夏季",
    9: "秋季", 10: "秋季", 11: "秋季",
}

print(f"仓库根目录 : {ROOT}")
print(f"清洗数据   : {PROCESSED.name} (存在: {PROCESSED.exists()})")

仓库根目录 : C:\Users\Jyvting\Desktop\power-load-forecast-topic4
清洗数据   : power_load_cleaned.csv (存在: True)


## 步骤 1　加载数据并核对样本边界

先确认行数、时间频率、缺失值和可观测日期，避免把“364 天”写成“全年 365 天”。


In [2]:
df = pd.read_csv(PROCESSED, index_col="datetime", parse_dates=True)
df.index.name = "datetime"

date_counts = df.groupby(df.index.normalize()).size()
audit = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "起始时间": df.index.min(),
    "结束时间": df.index.max(),
    "可观测日期数": df.index.normalize().nunique(),
    "时间频率": df.index.inferred_freq,
    "缺失值总数": int(df.isna().sum().sum()),
    "每日均为144条": bool((date_counts == 144).all()),
})
display(audit.to_frame("核对结果"))

print("前 3 行：")
display(df.head(3))
print("\n各列缺失值：")
display(df.isna().sum().to_frame("缺失数"))


,核对结果
行数,52416
列数,9
起始时间,2017-01-01 00:00:00
结束时间,2017-12-30 23:50:00
可观测日期数,364
时间频率,10min
缺失值总数,0
每日均为144条,True


前 3 行：


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power_consumption,zone2_power_consumption,zone3_power_consumption,zone_total_power_consumption
datetime,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.875380,20240.96386,70425.535440
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,17567.781155,20131.08434,67513.549035
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.686930,19668.43373,67803.221930



各列缺失值：


,缺失数
temperature,0
humidity,0
wind_speed,0
general_diffuse_flows,0
diffuse_flows,0
zone1_power_consumption,0
zone2_power_consumption,0
zone3_power_consumption,0
zone_total_power_consumption,0


## 步骤 2　时间特征工程

从 `datetime` 索引派生小时、星期、月份和季节。原数据保持只读，后续聚合只使用并行的 `feat`。


In [3]:
feat = pd.DataFrame(index=df.index)
feat["hour"] = df.index.hour
feat["dow"] = df.index.dayofweek
feat["month"] = df.index.month
feat["season"] = df.index.month.map(SEASON_MAP)

display(feat.head())
print("各维度取值数:", {k: int(feat[k].nunique()) for k in ["hour", "dow", "month", "season"]})


,hour,dow,month,season
datetime,,,,
2017-01-01 00:00:00,0,6,1,冬季
2017-01-01 00:10:00,0,6,1,冬季
2017-01-01 00:20:00,0,6,1,冬季
2017-01-01 00:30:00,0,6,1,冬季
2017-01-01 00:40:00,0,6,1,冬季


各维度取值数: {'hour': 24, 'dow': 7, 'month': 12, 'season': 4}


## 步骤 3　日内负荷规律

按小时对全部可观测日取均值，得到 24 个小时的典型日曲线。重点回答三件事：

1. 全天峰值和谷值分别出现在几点；
2. 是否存在午间平台，平台大致覆盖哪些小时；
3. 三个分区的峰谷时刻和幅度是否同步。


In [4]:
profile_cols = [TOTAL] + ZONE_COLS
hourly = df.groupby(feat["hour"])[profile_cols].mean()
hourly.index.name = "hour"
hourly.round(3).to_csv(TABLE_DIR / "hourly_load_profile.csv", encoding="utf-8-sig")

peak_hour = int(hourly[TOTAL].idxmax())
valley_hour = int(hourly[TOTAL].idxmin())
peak_value = float(hourly.loc[peak_hour, TOTAL])
valley_value = float(hourly.loc[valley_hour, TOTAL])
peak_to_valley = peak_value / valley_value

print(f"峰值小时 : {peak_hour}:00  {peak_value:,.1f} kW")
print(f"谷值小时 : {valley_hour}:00  {valley_value:,.1f} kW")
print(f"峰谷比   : {peak_to_valley:.3f}")
print("\n12-15 时午间平台均值 :", f"{hourly.loc[12:15, TOTAL].mean():,.1f} kW")
print("18-22 时晚间高峰均值 :", f"{hourly.loc[18:22, TOTAL].mean():,.1f} kW")

fig, ax = plt.subplots(figsize=(12, 5.2))
ax.plot(hourly.index, hourly[TOTAL], lw=2.8, color=COL_COLOR[TOTAL], label=COL_LABEL[TOTAL])
for col in ZONE_COLS:
    ax.plot(hourly.index, hourly[col], lw=1.7, color=COL_COLOR[col], label=COL_LABEL[col])
ax.axvline(peak_hour, color="#7f1d1d", ls="--", lw=1, alpha=0.8)
ax.axvline(valley_hour, color="#475569", ls=":", lw=1.2, alpha=0.8)
ax.scatter([peak_hour], [peak_value], color="#7f1d1d", zorder=5, s=35)
ax.scatter([valley_hour], [valley_value], color="#475569", zorder=5, s=35)
ax.annotate(f"峰值 {peak_hour}:00\n{peak_value:,.0f} kW", (peak_hour, peak_value),
            xytext=(8, -34), textcoords="offset points", fontsize=9, color="#7f1d1d")
ax.annotate(f"谷值 {valley_hour}:00\n{valley_value:,.0f} kW", (valley_hour, valley_value),
            xytext=(8, 10), textcoords="offset points", fontsize=9, color="#475569")
ax.set_xlabel("小时")
ax.set_ylabel("平均负荷（kW）")
ax.set_title("日内负荷曲线（2017 年可观测样本按小时平均）", loc="left")
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=9, ncol=2)
fig.savefig(FIG_DIR / "figA_daily_profile.png", dpi=180)
plt.close(fig)

print("\n各分区峰值/谷值：")
zone_peak = pd.DataFrame({
    "峰值小时": [int(hourly[c].idxmax()) for c in ZONE_COLS],
    "峰值(kW)": [float(hourly[c].max()) for c in ZONE_COLS],
    "谷值小时": [int(hourly[c].idxmin()) for c in ZONE_COLS],
    "谷值(kW)": [float(hourly[c].min()) for c in ZONE_COLS],
    "峰谷比": [float(hourly[c].max() / hourly[c].min()) for c in ZONE_COLS],
}, index=[COL_LABEL[c] for c in ZONE_COLS])
display(zone_peak.round(2))
zone_peak.round(6).to_csv(TABLE_DIR / "zone_peak_valley.csv", encoding="utf-8-sig")


峰值小时 : 20:00  98,041.2 kW
谷值小时 : 6:00  50,193.6 kW
峰谷比   : 1.953

12-15 时午间平台均值 : 75,130.2 kW
18-22 时晚间高峰均值 : 92,464.9 kW



各分区峰值/谷值：


,峰值小时,峰值(kW),谷值小时,谷值(kW),峰谷比
Zone1,20,43825.31,6,23205.38,1.89
Zone2,20,28186.82,6,14934.68,1.89
Zone3,20,26029.10,7,11670.85,2.23


### 日内规律结论

- 全天负荷在 **06:00 左右最低（约 50,194 kW）**，之后持续爬升，**12—14 时稳定在约 75,000—76,000 kW，15 时约 73,800 kW**，**20:00 达到全天峰值（约 98,041 kW）**，峰谷比约 1.95。
- 因此本数据是“**午间平台 + 晚间主峰**”，不是等高的“早晚双峰”；把 10:00 和 18:00 说成两个主峰与曲线不符。
- 三个分区的日内峰谷时刻基本同步：Zone1、Zone2 在 20:00 达峰、06:00 见谷；Zone3 也在 20:00 达峰，但谷值出现在 07:00。说明全城用电节奏一致，差异主要体现在幅度和波动性上。

## 步骤 4　周内负荷规律

按星期几取均值，比较工作日与周末。除总负荷外，分别计算三个分区的“周末/工作日”比值，判断周末效应主要来自哪个分区。


In [5]:
DOW_LABEL = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
weekly = df.groupby(feat["dow"])[profile_cols].mean().reindex(range(7))
weekly.index = DOW_LABEL
weekly.round(3).to_csv(TABLE_DIR / "weekly_load_profile.csv", encoding="utf-8-sig")

weekday_mean = float(weekly.loc[DOW_LABEL[:5], TOTAL].mean())
weekend_mean = float(weekly.loc[DOW_LABEL[5:], TOTAL].mean())
weekend_ratio = weekend_mean / weekday_mean

zone_weekend_ratio = pd.DataFrame({
    "工作日均值(kW)": [float(weekly.loc[DOW_LABEL[:5], c].mean()) for c in ZONE_COLS],
    "周末均值(kW)": [float(weekly.loc[DOW_LABEL[5:], c].mean()) for c in ZONE_COLS],
}, index=[COL_LABEL[c] for c in ZONE_COLS])
zone_weekend_ratio["周末/工作日"] = zone_weekend_ratio["周末均值(kW)"] / zone_weekend_ratio["工作日均值(kW)"]
zone_weekend_ratio["变化(%)"] = (zone_weekend_ratio["周末/工作日"] - 1) * 100
zone_weekend_ratio.round(4).to_csv(TABLE_DIR / "zone_weekend_ratio.csv", encoding="utf-8-sig")

print(weekly.round(1).rename(columns=COL_LABEL))
print(f"\n工作日总负荷均值 : {weekday_mean:,.1f} kW")
print(f"周末总负荷均值   : {weekend_mean:,.1f} kW")
print(f"周末/工作日      : {weekend_ratio:.4f}  ({(weekend_ratio - 1) * 100:+.2f}%)")
display(zone_weekend_ratio.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = ["#1f4e9c"] * 5 + ["#e07b39"] * 2
bars = axes[0].bar(weekly.index, weekly[TOTAL], color=bar_colors, alpha=0.9)
for b, v in zip(bars, weekly[TOTAL]):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 250, f"{v:,.0f}", ha="center", fontsize=8)
axes[0].set_ylim(weekly[TOTAL].min() * 0.93, weekly[TOTAL].max() * 1.04)
axes[0].set_ylabel("平均负荷（kW）")
axes[0].set_title("(a) 周内平均总负荷", loc="left")
axes[0].tick_params(axis="x", labelrotation=0)

ratio_values = zone_weekend_ratio["周末/工作日"].values
ratio_bars = axes[1].bar(zone_weekend_ratio.index, ratio_values, color=[COL_COLOR[c] for c in ZONE_COLS], alpha=0.9)
for b, v in zip(ratio_bars, ratio_values):
    axes[1].text(b.get_x() + b.get_width() / 2, v + 0.006, f"{v:.3f}", ha="center", fontsize=9)
axes[1].axhline(1.0, color="#334155", ls="--", lw=1)
axes[1].set_ylim(min(0.88, ratio_values.min() - 0.02), max(1.03, ratio_values.max() + 0.02))
axes[1].set_ylabel("周末均值 / 工作日均值")
axes[1].set_title("(b) 分区的周末效应", loc="left")
fig.suptitle("周内负荷规律与分区周末效应", x=0.01, ha="left", fontsize=13)
fig.savefig(FIG_DIR / "figB_weekly_profile.png", dpi=180)
plt.close(fig)


        总负荷    Zone1    Zone2    Zone3
周一  71706.6  32616.1  21280.4  17810.1
周二  72066.6  32716.2  21518.8  17831.6
周三  72352.5  32873.9  21627.2  17851.4
周四  72533.2  32844.3  21844.1  17844.9
周五  71347.9  32339.9  21461.1  17546.9
周六  70851.9  32255.0  20585.6  18011.3
周日  67713.6  30779.5  18978.9  17955.2

工作日总负荷均值 : 72,001.4 kW
周末总负荷均值   : 69,282.8 kW
周末/工作日      : 0.9622  (-3.78%)


,工作日均值(kW),周末均值(kW),周末/工作日,变化(%)
Zone1,32678.0632,31517.2067,0.9645,-3.5524
Zone2,21546.3158,19782.2705,0.9181,-8.1872
Zone3,17776.9725,17983.2900,1.0116,1.1606


### 周内规律结论

- 工作日总负荷均值约 **72,001 kW**，周末约 **69,283 kW**，周末/工作日为 **0.962**，整体只低 **3.8%**。因此不宜写成“周末显著低于工作日”。
- 周末效应主要来自 **Zone2**：其周末/工作日比值为 **0.918（低 8.2%）**；Zone1 为 0.964（低 3.6%）；Zone3 为 1.012（基本持平）。
- 这说明该城市的周节律不是三个分区同步减弱，而是 **Zone2 更接近工作日驱动的用电结构，Zone3 对周末不敏感**。报告应把这个分区差异写出来，而不是只给一条总负荷曲线。

## 步骤 5　季节负荷规律

按月与按季聚合。季节划分：春 3-5 月、夏 6-8 月、秋 9-11 月、冬 12 月及 1-2 月。


In [6]:
monthly = df.groupby(feat["month"])[profile_cols].mean()
seasonal = df.groupby(feat["season"])[profile_cols].mean().reindex(SEASON_ORDER)
monthly.round(3).to_csv(TABLE_DIR / "monthly_load_mean.csv", encoding="utf-8-sig")
seasonal.round(3).to_csv(TABLE_DIR / "seasonal_load_mean.csv", encoding="utf-8-sig")

annual_mean = float(df[TOTAL].mean())
summer_mean = float(seasonal.loc["夏季", TOTAL])
winter_mean = float(seasonal.loc["冬季", TOTAL])
print(seasonal.round(1).rename(columns=COL_LABEL))
print(f"\n全年样本均值 : {annual_mean:,.1f} kW")
print(f"夏季均值     : {summer_mean:,.1f} kW  (相对全年 {summer_mean / annual_mean - 1:+.1%})")
print(f"冬季均值     : {winter_mean:,.1f} kW  (相对夏季 {winter_mean / summer_mean - 1:+.1%})")
print(f"最高月份     : {int(monthly[TOTAL].idxmax())} 月  {monthly[TOTAL].max():,.1f} kW")
print(f"最低月份     : {int(monthly[TOTAL].idxmin())} 月  {monthly[TOTAL].min():,.1f} kW")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for col in [TOTAL] + ZONE_COLS:
    axes[0].plot(monthly.index, monthly[col], lw=2, marker="o", color=COL_COLOR[col], label=COL_LABEL[col])
axes[0].set_xlabel("月份")
axes[0].set_ylabel("平均负荷（kW）")
axes[0].set_title("(a) 月度平均负荷", loc="left")
axes[0].set_xticks(range(1, 13))
axes[0].legend(fontsize=8, ncol=2)

x = np.arange(len(SEASON_ORDER))
w = 0.25
for i, col in enumerate(ZONE_COLS):
    axes[1].bar(x + (i - 1) * w, seasonal[col], w, color=COL_COLOR[col], label=COL_LABEL[col])
axes[1].set_xticks(x)
axes[1].set_xticklabels(SEASON_ORDER)
axes[1].set_ylabel("平均负荷（kW）")
axes[1].set_title("(b) 季度平均负荷（分区）", loc="left")
axes[1].legend(fontsize=8)
fig.suptitle("月度与季度负荷规律", x=0.01, ha="left", fontsize=13)
fig.savefig(FIG_DIR / "figC_seasonal_profile.png", dpi=180)
plt.close(fig)


            总负荷    Zone1    Zone2    Zone3
season                                    
春季      67991.5  31579.0  18700.9  17711.7
夏季      83289.8  35636.9  23185.0  24468.0
秋季      67063.9  31755.7  21628.4  13679.8
冬季      66348.9  30342.2  20648.5  15358.2

全年样本均值 : 71,224.6 kW
夏季均值     : 83,289.8 kW  (相对全年 +16.9%)
冬季均值     : 66,348.9 kW  (相对夏季 -20.3%)
最高月份     : 7 月  88,176.0 kW
最低月份     : 12 月  63,751.8 kW


### 季节规律结论

- 四个季度的总负荷均值分别为：春季约 67,992 kW、夏季约 83,290 kW、秋季约 67,064 kW、冬季约 66,349 kW。
- 夏季比全年样本均值高约 **16.9%**，比冬季高约 **25.5%**；7 月最高（约 88,176 kW），12 月最低（约 63,752 kW）。
- 冬季虽然最低，但仅比秋季低 1.1%、比春季低 2.5%，因此更准确的表述是“**夏季单峰主导，其他三季接近**”，而不是“夏冬双峰”。
- 夏季负荷抬升与高温同期出现，提示降温负荷是重要因素；但仅凭这份数据不能把全部增量归因于空调，仍需结合产业结构、节假日和生产安排解释。

## 步骤 6　三分区用电特征对比

从四个角度比较三个分区：

1. 规模与占比；
2. 波动性（标准差、变异系数、峰谷比、日内标准差）；
3. 周内敏感性（周末/工作日比值）；
4. 结构变化（Zone3/Zone1 月度比值）。


In [7]:
zone_metrics = pd.DataFrame(index=ZONE_COLS)
zone_metrics["平均负荷(kW)"] = df[ZONE_COLS].mean()
zone_metrics["占比(%)"] = zone_metrics["平均负荷(kW)"] / df[TOTAL].mean() * 100
zone_metrics["标准差(kW)"] = df[ZONE_COLS].std()
zone_metrics["变异系数"] = zone_metrics["标准差(kW)"] / zone_metrics["平均负荷(kW)"]
zone_metrics["P95(kW)"] = df[ZONE_COLS].quantile(0.95)
zone_metrics["最大值(kW)"] = df[ZONE_COLS].max()
zone_metrics["峰值小时"] = [int(hourly[c].idxmax()) for c in ZONE_COLS]
zone_metrics["谷值小时"] = [int(hourly[c].idxmin()) for c in ZONE_COLS]
zone_metrics["峰谷比"] = [float(hourly[c].max() / hourly[c].min()) for c in ZONE_COLS]
zone_metrics["日内标准差均值(kW)"] = df[ZONE_COLS].groupby(df.index.normalize()).std().mean()
zone_metrics.index = [COL_LABEL[c] for c in ZONE_COLS]
zone_metrics["周末/工作日"] = zone_weekend_ratio["周末/工作日"]
zone_metrics.round(4).to_csv(TABLE_DIR / "zone_profile_metrics.csv", encoding="utf-8-sig")
pd.DataFrame({
    "平均负荷(kW)": zone_metrics["平均负荷(kW)"],
    "占总负荷(%)": zone_metrics["占比(%)"],
}).round(6).to_csv(TABLE_DIR / "zone_share.csv", encoding="utf-8-sig", index_label="分区")

print("=== 三分区用电特征指标 ===")
display(zone_metrics.round(3))

zone_corr = df[ZONE_COLS].corr()
zone_corr.index = [COL_LABEL[c] for c in ZONE_COLS]
zone_corr.columns = [COL_LABEL[c] for c in ZONE_COLS]
zone_corr.round(4).to_csv(TABLE_DIR / "zone_corr.csv", encoding="utf-8-sig")
print("\n=== 三分区负荷相关系数 ===")
display(zone_corr.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([df[c].values for c in ZONE_COLS], widths=0.55, patch_artist=True,
                showfliers=True, whis=1.5)
for patch, col in zip(bp["boxes"], [COL_COLOR[c] for c in ZONE_COLS]):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)
ax.set_xticklabels([COL_LABEL[c] for c in ZONE_COLS])
ax.set_ylabel("负荷（kW）")
ax.set_title("三分区负荷分布对比（箱线图）", loc="left")
fig.savefig(FIG_DIR / "figD_zone_boxplot.png", dpi=180)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
means = zone_metrics["平均负荷(kW)"].values
shares = zone_metrics["占比(%)"].values
bars = axes[0].bar(zone_metrics.index, means, color=[COL_COLOR[c] for c in ZONE_COLS], alpha=0.9)
for b, v, s in zip(bars, means, shares):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 500, f"{v:,.0f}\n({s:.1f}%)", ha="center", fontsize=9)
axes[0].set_ylim(0, max(means) * 1.16)
axes[0].set_ylabel("平均负荷（kW）")
axes[0].set_title("(a) 规模与占比", loc="left")

x = np.arange(len(ZONE_COLS))
axes[1].bar(x - 0.19, zone_metrics["变异系数"], 0.36, color="#64748b", label="变异系数")
axes[1].bar(x + 0.19, zone_metrics["峰谷比"], 0.36, color="#0f766e", label="峰谷比")
axes[1].set_xticks(x)
axes[1].set_xticklabels(zone_metrics.index)
axes[1].set_title("(b) 波动性指标", loc="left")
axes[1].legend(fontsize=8)
fig.suptitle("三分区规模与波动性对比", x=0.01, ha="left", fontsize=13)
fig.savefig(FIG_DIR / "figE_zone_metrics.png", dpi=180)
plt.close(fig)

monthly_zone = df.groupby(feat["month"])[ZONE_COLS].mean()
ratio = monthly_zone[ZONE_COLS[2]] / monthly_zone[ZONE_COLS[0]]
ratio_df = pd.DataFrame({
    "Zone1均值(kW)": monthly_zone[ZONE_COLS[0]],
    "Zone3均值(kW)": monthly_zone[ZONE_COLS[2]],
    "Zone3/Zone1": ratio,
})
ratio_df.index.name = "month"
ratio_df.round(6).to_csv(TABLE_DIR / "zone3_ratio_monthly.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(ratio.index, ratio.values, "o-", color="#c0392b", lw=2.2)
ax.axhline(1.0, color="gray", ls="--", lw=1)
ax.set_xticks(range(1, 13))
ax.set_xlabel("月份")
ax.set_ylabel("Zone3 / Zone1")
ax.set_title("Zone3 相对 Zone1 的月度比值（结构变化）", loc="left")
for m in [7, 9, 12]:
    ax.annotate(f"{ratio.loc[m]:.3f}", (m, ratio.loc[m]), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=9)
fig.savefig(FIG_DIR / "figF_zone3_ratio.png", dpi=180)
plt.close(fig)

print(f"\nZone3/Zone1：7 月 {ratio.loc[7]:.3f}，9 月 {ratio.loc[9]:.3f}，12 月 {ratio.loc[12]:.3f}")
print(f"7 -> 12 月比值变化：{(ratio.loc[12] / ratio.loc[7] - 1) * 100:+.1f}%")


=== 三分区用电特征指标 ===


,平均负荷(kW),占比(%),标准差(kW),变异系数,P95(kW),最大值(kW),峰值小时,谷值小时,峰谷比,日内标准差均值(kW),周末/工作日
Zone1,32346.390,45.415,7128.629,0.220,44712.911,52204.395,20,6,1.889,6609.931,0.964
Zone2,21042.303,29.544,5200.327,0.247,30377.107,36645.570,20,6,1.887,4399.608,0.918
Zone3,17835.920,25.042,6621.026,0.371,29905.709,47598.326,20,7,2.230,4314.167,1.012



=== 三分区负荷相关系数 ===


,Zone1,Zone2,Zone3
Zone1,1.000,0.835,0.751
Zone2,0.835,1.000,0.571
Zone3,0.751,0.571,1.000



Zone3/Zone1：7 月 0.787，9 月 0.447，12 月 0.381
7 -> 12 月比值变化：-51.6%


### 三分区对比结论

- **规模**：Zone1 平均约 32,346 kW，占 45.4%；Zone2 约 21,042 kW，占 29.5%；Zone3 约 17,836 kW，占 25.0%。Zone1 是基数最大的分区。
- **波动性**：Zone3 的变异系数为 0.371、峰谷比为 2.23，均高于 Zone1（0.220、1.89）和 Zone2（0.247、1.89）。因此“波动最剧烈”的是 **Zone3**，不是 Zone1。
- **周内敏感性**：Zone2 的周末/工作日比值最低（0.918），受工作日作息影响最明显；Zone1 为 0.964；Zone3 为 1.012，几乎没有周末效应。
- **结构变化**：Zone3/Zone1 的月度比值从 7 月的 0.787 降至 12 月的 0.381，8 月后总体走低（11 月小幅回升）。它更像持续的结构变化，而不是单点异常，因此不应按普通离群点抹平。仅凭现有字段无法判断原因，报告只能写“可能与 Zone3 大用户生产调整有关，需业务核实”，不能直接断言“停限产”。
- **分区相关性**：三个分区负荷高度同向变化，说明全城用电节奏一致；但幅度、波动和周末敏感性不同，调度上应区别对待。

## 步骤 7　小结与交付清单

| 题目要求 | 本 notebook 对应 |
|---|---|
| 日尺度负荷规律 | 报告图 1、`tables/hourly_load_profile.csv`、`tables/zone_peak_valley.csv` |
| 周尺度负荷规律 | 报告图 2、`tables/weekly_load_profile.csv`、`tables/zone_weekend_ratio.csv` |
| 季节尺度负荷规律 | 报告图 3、`tables/monthly_load_mean.csv`、`tables/seasonal_load_mean.csv` |
| 三分区用电特征对比 | 报告图 4—6、`tables/zone_profile_metrics.csv`、`tables/zone_corr.csv`、`tables/zone3_ratio_monthly.csv` |

### 给谭煜霖（成品图）和沈成源（建模）的接口

- 谭煜霖重绘图时保留“峰值/谷值、周末/工作日、变异系数/峰谷比”这些标注，不要只画平滑曲线。
- 沈成源在 03 中至少应加入小时、星期、月份/季节特征；若使用气象变量，应考虑月份与小时的交互，因为相关强度随时段变化。
- 报告写 2017 年样本时统一写 364 天，避免 12 月 31 日缺失造成口径不一致。